In [6]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
BASE = Path().resolve().parent
DATA = BASE / 'data'
OUTPUTS = BASE / 'outputs'

np.random.seed(24)
random.seed(24)

# Channel Configuration
# Each channel has: volume share, conversion rates per stage, CPC (cost per click)
CHANNELS = {
    "Google Ads": {
        "weight": 0.35,
        "cpc": 120,              # ₹ cost per click
        "land_to_signup": 0.38,
        "signup_to_trial": 0.55,
        "trial_to_paid": 0.17,
    },
    "Facebook Ads": {
        "weight": 0.25,
        "cpc": 65,
        "land_to_signup": 0.28,
        "signup_to_trial": 0.48,
        "trial_to_paid": 0.12,   # lowest — broad audience, low intent
    },
    "YouTube Ads": {
        "weight": 0.15,
        "cpc": 45,
        "land_to_signup": 0.32,
        "signup_to_trial": 0.52,
        "trial_to_paid": 0.16,
    },
    "Organic SEO": {
        "weight": 0.17,
        "cpc": 0,
        "land_to_signup": 0.44,
        "signup_to_trial": 0.62,
        "trial_to_paid": 0.22,
    },
    "Referral": {
        "weight": 0.08,
        "cpc": 20,
        "land_to_signup": 0.58,
        "signup_to_trial": 0.72,
        "trial_to_paid": 0.31,
    },
}
N_AD_IMPRESSIONS = 10_000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)
AVG_REVENUE_PER_PAID_USER = 4500 # average course price

### Generate the full funnel dataset.

In [7]:
def random_date(start, end):
    delta = end - start
    return start + timedelta(
        days=random.randint(0, delta.days),
        hours=random.randint(8, 22),
        minutes=random.randint(0, 59)
    )

rows = []
channel_names = list(CHANNELS.keys())
channel_weights = [CHANNELS[c]['weight'] for c in channel_names]

for i in range(N_AD_IMPRESSIONS):
    user_id = f"USR-{str(i+1).zfill(3)}"
    channel = random.choices(channel_names, weights=channel_weights)[0]
    cfg = CHANNELS[channel]

    ad_date = random_date(START_DATE, END_DATE)

    # stage 1 -> 2: did user click the ad and reach landing page?
    # ~65% of ad impression results in a landing page visit (click-through)
    reached_landing = random.random() < 0.65

    # stage 2->3: did user sign up from landing page?
    signed_up = reached_landing and (random.random() < cfg['land_to_signup'])

    # stage 3->4: did user start a trail?
    started_trial = signed_up and (random.random() < cfg['signup_to_trial'])

    # stage 4->5: did user convert to paid?
    converted_paid = started_trial and (random.random() < cfg['trial_to_paid'])

    # Time between stages
    signup_date = ad_date + timedelta(
        minutes=random.randint(5, 120))
    trial_date = signup_date + timedelta(
        hours=random.randint(1, 48))
    paid_date = trial_date + timedelta(
        days = random.randint(1, 21))

    # Revenue - some users buy premium (higher price)
    revenue = 0
    if converted_paid:
        revenue = random.choices(
            [2999, 4999, 7999, 12999],
            weights=[0.30, 0.40, 0.20, 0.10]
        )[0]

    # device type -
    device = random.choices(['Mobile', 'Desktop', 'Tablet'],
                            weights=[0.55, 0.38, 0.07])[0]
     # Course category interest
    course = random.choices(
        ["Data Science", "Web Dev", "Digital Marketing",
         "Finance", "Design"],
        weights=[0.30, 0.25, 0.20, 0.15, 0.10]
    )[0]

    rows.append({
        "user_id":          user_id,
        "channel":          channel,
        "device":           device,
        "course_interest":  course,
        "ad_date":          ad_date,
        "reached_landing":  reached_landing,
        "signed_up":        signed_up,
        "started_trial":    started_trial,
        "converted_paid":   converted_paid,
        "signup_date":      signup_date,
        "trial_date":       trial_date,
        "paid_date":        paid_date,
        "revenue":          revenue,
        "cpc":              cfg["cpc"],
    })

In [8]:
df = pd.DataFrame(rows)
df.to_csv(f"{DATA}/funnel_events.csv", index=False)
print(f"Generated: {len(df):,} users")
print(f"Paid conversions: {df['converted_paid'].sum():,}")
print(f"Overall conversion rate: {df['converted_paid'].mean()*100:.2f}%")
print(f"Total revenue: ₹{df['revenue'].sum():,.0f}")

Generated: 10,000 users
Paid conversions: 251
Overall conversion rate: 2.51%
Total revenue: ₹1,464,749
